### Load Data

In [2]:
import os
import numpy as np

# ✅ Path to your saved .npy file
npy_file_path = "../MediaPipe_landmarks/squat_back_new_landmarks.npy"

# ✅ Load the NumPy array
landmarks_data = np.load(npy_file_path)

# ✅ Check its shape
print("Shape:", landmarks_data.shape)

# ✅ Inspect first frame
print("First frame landmarks:\n", landmarks_data)


Shape: (742, 33, 3)
First frame landmarks:
 [[[ 0.45064944  0.36396754  0.17328146]
  [ 0.44843626  0.35616517  0.16817467]
  [ 0.44696623  0.35604924  0.1681011 ]
  ...
  [ 0.46197635  0.79510498 -0.02590076]
  [ 0.4363803   0.79401076 -0.01820946]
  [ 0.47002825  0.79102546 -0.05297839]]

 [[ 0.4506503   0.36336634  0.16061236]
  [ 0.44833249  0.35552183  0.15250959]
  [ 0.4468579   0.3553212   0.15244085]
  ...
  [ 0.46014279  0.79514694 -0.02586933]
  [ 0.43671101  0.79308593 -0.00983211]
  [ 0.47017464  0.79106367 -0.05263109]]

 [[ 0.4505659   0.36140192  0.1524386 ]
  [ 0.44801071  0.35407662  0.14435907]
  [ 0.44644299  0.35404211  0.14429715]
  ...
  [ 0.45985621  0.79499245 -0.02241618]
  [ 0.4367345   0.79298121 -0.00654933]
  [ 0.47018316  0.7910651  -0.04587119]]

 ...

 [[ 0.47704986  0.39920992  0.20848788]
  [ 0.47515395  0.39309323  0.20082073]
  [ 0.47373477  0.39288527  0.2007487 ]
  ...
  [ 0.48691025  0.78535026 -0.10753024]
  [ 0.43673748  0.76754844 -0.0392955 ]


### Index different joints and normalize skeleton to be centered by the pelvis and size of entire skeleton 

In [3]:
import sys, os

# Go two levels up to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
sys.path.append(project_root)

print("Project root added to path:", project_root)

from Utils.utils.utils import *



# MediaPipe joint indices
HIP_L = 23
KNEE_L = 25
ANKLE_L = 27
TOE_L = 31
HEEL_L = 29

HIP_R = 24
KNEE_R = 26
ANKLE_R = 28
TOE_R = 32
HEEL_R = 30

SHOULDER_L = 11
ELBOW_L = 13
WRIST_L = 15
THUMB_L = 21
INDEXFINGER_L = 19
PINKY_L = 17

SHOULDER_R = 12
ELBOW_R = 14
WRIST_R = 16
THUMB_R = 22
INDEXFINGER_R = 20
PINKY_R = 18

NOSE = 0



def normalize_skeleton_with_virtual_joints(coords, lhip_idx, rhip_idx,
                                           lsho_idx, rsho_idx, eps=1e-8):
    """
    coords: (T, J, 3) raw 3D landmarks
    lhip_idx, rhip_idx: left/right hip indices
    lsho_idx, rsho_idx: left/right shoulder indices

    Returns:
      coords_norm: (T, J+2, 3) normalized coords including virtual pelvis and neck
      pelvis:      (T, 3) pelvis positions before centering
      neck:        (T, 3) neck positions before centering
    """

    T, J, _ = coords.shape

    # 1) Virtual mid-pelvis and mid-shoulder (neck proxy)
    left_hip  = coords[:, lhip_idx, :]    # (T, 3)
    right_hip = coords[:, rhip_idx, :]    # (T, 3)
    pelvis = (left_hip + right_hip) / 2.0 # (T, 3)

    left_sho  = coords[:, lsho_idx, :]    # (T, 3)
    right_sho = coords[:, rsho_idx, :]    # (T, 3)
    neck = (left_sho + right_sho) / 2.0   # (T, 3)

    # 2) Center all original joints on pelvis
    coords_centered = coords - pelvis[:, None, :]  # (T, J, 3)

    # 3) Also center virtual joints
    pelvis_centered = pelvis - pelvis              # becomes (T, 3) at origin
    neck_centered   = neck - pelvis                # neck relative to pelvis

    # 4) Stack virtual joints at the end: [J joints, pelvis, neck]
    pelvis_centered = pelvis_centered[:, None, :]  # (T, 1, 3)
    neck_centered   = neck_centered[:, None, :]    # (T, 1, 3)
    coords_with_virtual = np.concatenate(
        [coords_centered, pelvis_centered, neck_centered], axis=1
    )  # (T, J+2, 3)

    # 5) Compute scale as pelvis->neck distance
    # neck is last joint index: J+1
    neck_rel = coords_with_virtual[:, -1, :]                # (T, 3)
    scale = np.linalg.norm(neck_rel, axis=-1, keepdims=True) + eps  # (T, 1)

    # 6) Scale all joints
    coords_norm = coords_with_virtual / scale[:, None, :]   # (T, J+2, 3)

    return coords_norm, pelvis, neck



coords_norm, pelvis_raw, neck_raw = normalize_skeleton_with_virtual_joints(
    landmarks_data, HIP_L, HIP_R, SHOULDER_L, SHOULDER_R
)

Project root added to path: c:\Users\chris\OneDrive\Desktop\Fritidsprojekt\TempAISpotter\AI\OlympicAi


### Transform landmark coordinates into angles of different joints. We use angles for our embedding

In [4]:
def compute_angle_features(landmarks):
    frames, joints, dims = landmarks.shape
    features = []

    for f in range(frames):
        lm = landmarks[f]

        # Example angles
        left_ankle = calculate_angle(lm[KNEE_L], lm[ANKLE_L], lm[TOE_L])
        right_ankle = calculate_angle(lm[KNEE_R], lm[ANKLE_R], lm[TOE_R])
        
        left_knee = calculate_angle(lm[HIP_L], lm[KNEE_L], lm[ANKLE_L])
        right_knee = calculate_angle(lm[HIP_R], lm[KNEE_R], lm[ANKLE_R])
        
        left_hip = calculate_angle(lm[SHOULDER_L], lm[HIP_L], lm[KNEE_L])
        right_hip = calculate_angle(lm[SHOULDER_R], lm[HIP_R], lm[KNEE_R])
        
        left_shoulder = calculate_angle(lm[ELBOW_L], lm[SHOULDER_L], lm[HIP_L])
        right_shoulder = calculate_angle(lm[ELBOW_R], lm[SHOULDER_R], lm[HIP_R])

        left_elbow = calculate_angle(lm[SHOULDER_L], lm[ELBOW_L], lm[WRIST_L])
        right_elbow = calculate_angle(lm[SHOULDER_R], lm[ELBOW_R], lm[WRIST_R])
        
        left_wrist = calculate_angle(lm[ELBOW_L], lm[WRIST_L], lm[PINKY_L])
        right_wrist = calculate_angle(lm[ELBOW_R], lm[WRIST_R], lm[PINKY_R])

        # Add more angles if you want a richer embedding

        features.append([
            left_ankle,
            right_ankle,
            left_knee,
            right_knee,
            left_hip,
            right_hip,
            left_shoulder,
            right_shoulder,
            left_wrist,
            right_wrist,
            right_elbow,
            left_elbow,
        ])

    return np.array(features)

new_arr = compute_angle_features(landmarks_data)
new_arr.shape

(742, 12)

In [5]:
new_arr

array([[173.39960894, 165.07162933, 177.86105137, ..., 175.28570405,
        173.35736401, 169.41352812],
       [168.07339765, 151.69591624, 178.35217047, ..., 175.04697073,
        172.13746591, 174.26792544],
       [166.4135722 , 149.08536345, 178.61431974, ..., 173.89313418,
        172.38953449, 173.38387565],
       ...,
       [163.66823372, 142.02937216, 176.35302207, ..., 166.09049465,
         27.16563099,  32.75494832],
       [163.37072059, 142.30591565, 176.28641892, ..., 167.78985549,
         27.49737783,  32.6695756 ],
       [163.02060063, 142.55722534, 176.2132976 , ..., 167.98647707,
         27.77427081,  32.59752549]])

In [6]:
new_arr.shape

(742, 12)

### Normalize values from pure angles to (something that i need to check what it gets turned into)

In [7]:
norm_arr = (new_arr - new_arr.mean(axis=0)) / new_arr.std(axis=0)
norm_arr

array([[ 0.19931246,  1.90243219,  0.19331443, ...,  0.51122712,
         3.56194469,  4.34707237],
       [-0.13994239,  0.91653626,  0.21799624, ...,  0.50334489,
         3.52857064,  4.51284587],
       [-0.24566554,  0.72411786,  0.23117088, ...,  0.4652488 ,
         3.53546675,  4.48265633],
       ...,
       [-0.42053081,  0.20403545,  0.11752651, ...,  0.20762997,
        -0.43757709, -0.31970095],
       [-0.43948101,  0.22441889,  0.11417929, ...,  0.26373757,
        -0.42850114, -0.32261635],
       [-0.46178202,  0.24294241,  0.11050448, ...,  0.2702294 ,
        -0.42092589, -0.3250768 ]])

### Add a feature which tells the velocity of movement. Used to compare speed of reps

In [8]:
velocity = np.diff(norm_arr, axis=0)
velocity

array([[-3.39254846e-01, -9.85895931e-01,  2.46818090e-02, ...,
        -7.88223028e-03, -3.33740418e-02,  1.65773506e-01],
       [-1.05723148e-01, -1.92418404e-01,  1.31746421e-02, ...,
        -3.80960869e-02,  6.89610654e-03, -3.01895422e-02],
       [-1.10150232e-01, -1.40691389e-01,  5.81376266e-03, ...,
        -2.03131757e-02,  5.94181889e-03,  3.21595841e-02],
       ...,
       [-1.24488642e-02,  2.60277920e-02, -1.22419461e-03, ...,
         1.04917357e-01,  2.46348507e-03, -3.00677218e-04],
       [-1.89502002e-02,  2.03834442e-02, -3.34722532e-03, ...,
         5.61075987e-02,  9.07594895e-03, -2.91540495e-03],
       [-2.23010102e-02,  1.85235134e-02, -3.67480422e-03, ...,
         6.49183168e-03,  7.57525403e-03, -2.46044960e-03]])

### Add feature of knee and hip symmetry

In [9]:
left_knee  = norm_arr[:, 3]
right_knee = norm_arr[:, 4]
left_hip   = norm_arr[:, 5]
right_hip  = norm_arr[:, 6]



knee_symmetry = left_knee - right_knee
hip_symmetry  = left_hip - right_hip
knee_symmetry.shape, hip_symmetry.shape

((742,), (742,))

### Smoothed the angles to create less noise in the data. (Maybe this is only relevant if we feed it through a ML pipeline?)

In [10]:
from scipy.signal import savgol_filter
angles_smooth = savgol_filter(norm_arr, window_length=11, polyorder=3, axis=0)
angles_smooth

array([[ 0.24433662,  1.78579241,  0.19542094, ...,  0.52514158,
         3.55237832,  4.38252577],
       [-0.12329746,  1.1404346 ,  0.21669374, ...,  0.48446457,
         3.54769795,  4.44596718],
       [-0.33542735,  0.71607682,  0.22784705, ...,  0.45923842,
         3.53902066,  4.49237225],
       ...,
       [-0.41876222,  0.20085398,  0.11671558, ...,  0.16771224,
        -0.43575316, -0.3205893 ],
       [-0.43706032,  0.22321581,  0.11398514, ...,  0.232042  ,
        -0.43067292, -0.32227174],
       [-0.46386812,  0.24486645,  0.11082508, ...,  0.30092826,
        -0.42058755, -0.32492985]])

### Make a new embedding with the new features we created

In [11]:
min_frames = velocity.shape[0]  # 415

# Remove last frames to make vectors match in dimensions
angles_smooth_trimmed = angles_smooth[:min_frames]
knee_symmetry_trimmed = knee_symmetry[:min_frames]
hip_symmetry_trimmed  = hip_symmetry[:min_frames]

# Add dimension for concatenation
knee_symmetry_trimmed = knee_symmetry_trimmed.reshape(-1, 1)
hip_symmetry_trimmed  = hip_symmetry_trimmed.reshape(-1, 1)


velocity.shape, angles_smooth_trimmed.shape, knee_symmetry_trimmed.shape, hip_symmetry_trimmed.shape

embedding = np.concatenate([
    angles_smooth_trimmed,
    velocity,                  # already 415
    knee_symmetry_trimmed,
    hip_symmetry_trimmed
], axis=1)

embedding.shape

(741, 26)

In [12]:
np.save("../embedding/squat_back_new_norm_embedding.npy", embedding)

In [13]:
bob = np.load("../embedding/squat_back_new_norm_embedding.npy")
bobby = np.load("../embedding/squat_back_res_bob_angle_embedding.npy")

In [14]:
bob.shape, bobby.shape

((741, 26), (415, 26))

### Use DTW to compare similarity of videos (figure out if i should use cosine, euclidean or manhatten distance metric)

In [15]:
from dtw import dtw
from scipy.spatial.distance import cosine, euclidean
dist, cost, acc, path = dtw(bob, bobby, dist=lambda x, y: cosine(x, y))
dist, path

(494.7266865198643,
 (array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
          13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
          26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
          39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
          52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
          65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
          78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
          91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
         104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
         117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
         130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
         143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
         156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
         169, 170,

### Map which frames correspond to which frame between the two videos

In [16]:
normalized_dist = dist / len(path[0])

#similarity = 1 - normalized_dist
normalized_dist

0.6676473502292366

In [17]:
print(path[0])  # Indices in bob
print(path[1])  # Indices in bobby

[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179
 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197
 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215
 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233
 234 235 236 237 238 239 240 241 242 243 244 245 24

### Calculate per-feature differences. Used to create data which can be used to find the specific deviations between videos (eg not deep enough squat based on not low enough knee angle)

In [18]:
idx_a, idx_b = path


aligned_diffs = []

for i, j in zip(idx_a, idx_b):
    diff = bob[i] - bobby[j]   # signed difference
    aligned_diffs.append(diff)

aligned_diffs = np.array(aligned_diffs)
# shape: (path_length, D)
aligned_diffs

array([[ 1.12815933,  2.34735831, -0.45534585, ...,  0.10867738,
         0.50428487,  2.02481263],
       [ 0.44263662,  1.40541642, -0.43578033, ..., -0.02119291,
         0.43475256,  1.95289853],
       [ 0.01641652,  0.76845108, -0.42520148, ...,  0.06439066,
         0.38963342,  1.85459899],
       ...,
       [-3.00858127, -1.23223464, -0.39831932, ...,  0.09523707,
         0.7324502 ,  0.35484738],
       [-3.03161742, -1.22715762, -0.39668462, ..., -0.11010321,
         0.70125237,  0.18750525],
       [-3.02993076, -1.20848966, -0.39684989, ..., -0.12715172,
         0.68293294,  0.08669402]])

### Find which feature contribute most to the error

In [19]:
feature_error = np.mean(np.abs(aligned_diffs), axis=0)

feature_error

array([0.76017462, 0.85684402, 0.6746436 , 0.3254956 , 0.70340694,
       0.34493224, 1.07127602, 0.8834006 , 0.88917965, 0.92716067,
       0.91738056, 0.98807335, 0.18949828, 0.18454737, 0.06345266,
       0.05143515, 0.07088534, 0.05553365, 0.17630244, 0.18606242,
       0.28338399, 0.28925241, 0.16312498, 0.15244084, 0.51535876,
       1.24401836])

### Find out when the difference occurs. Can be used to plot the difference in time.

In [20]:
knee_diff_over_time = np.abs(
    aligned_diffs[:, [0, 1]]
).mean(axis=1)


In [21]:
peak_idx = np.argmax(knee_diff_over_time)
peak_idx

175

In [22]:
bob_frame    = idx_a[peak_idx]
bobby_frame = idx_b[peak_idx]
bob_frame, bobby_frame

(175, 73)